# Alibaba 2017 - LSTM - next-interval forecast of `mem_util`

**Module:** time-series forecasting &nbsp;|&nbsp; **Dataset:** Alibaba 2017 &nbsp;|&nbsp; **Target:** `mem_util` (machine utilisation, next 5-minute interval)

**Input file:** `server_usage.csv` (genuine machine-level telemetry, 5-min cadence, 12 h)

**Pipeline:** load server_usage -> per machine: chronological order -> per-machine chronological 80/20 split -> `MinMaxScaler` fit on TRAIN points only -> transform test -> previous-24-interval windows (no window crosses the split).

**Architecture:** 64 -> 32 recurrent units, Dropout 0.2, Dense(1) &nbsp;|&nbsp; **Optimizer:** Adam(1e-3) &nbsp;|&nbsp; **Loss:** Huber(delta=1.0)

**Callbacks:** EarlyStopping(patience=8, restore_best_weights), ReduceLROnPlateau(factor=0.5, patience=4) &nbsp;|&nbsp; 30 epochs, batch 128, `validation_split=0.1`, `shuffle=False`

**Metrics (test):** MAE, RMSE, R2, sMAPE (train metrics also printed)

**Result row -> ** `results/sequence/Alibaba_2017_LSTM_Memory.csv`  (aggregated into `results/sequence_results.csv`)

> TensorFlow CPU training is not bit-for-bit deterministic; R2 varies by ~+-0.05 between runs. This is the same code that produced `results/sequence_results.csv`.

In [ ]:
# Step 1: Import libraries
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, SimpleRNN, Bidirectional, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
warnings.filterwarnings("ignore")

SEED = 42
tf.random.set_seed(SEED); np.random.seed(SEED)
EPS = 1e-9
WINDOW, HORIZON = 24, 1        # previous 24 five-minute intervals -> next interval
RESULTS_DIR = r"../../results"
UTIL_COL = "mem_util"        # TARGET

# Step 2: Load Alibaba server_usage  (machine-level telemetry)
ALIBABA_DIR = r"../../data/alibaba_2017"
su = pd.read_csv(os.path.join(ALIBABA_DIR, "server_usage.csv"), header=None,
                 names=["ts", "machine_id", "cpu_util", "mem_util", "disk_util",
                        "load1", "load5", "load15"])
su = su.dropna(subset=[UTIL_COL])
print("server_usage:", su.shape, "| machines:", su["machine_id"].nunique())

# Step 3-4: select the time-series per machine + chronological ordering
#           one series per machine, sorted by timestamp, rescaled percent -> [0,1]
MIN_LEN = 40
series = []
for _machine, g in su.sort_values(["machine_id", "ts"]).groupby("machine_id"):
    v = g[UTIL_COL].to_numpy(float) / 100.0
    if len(v) >= MIN_LEN:
        series.append(v)
print("machine series kept:", len(series))

# Step 5: sequence/window creation helper (previous WINDOW steps -> next step)
def make_windows(a, w=WINDOW, h=HORIZON):
    X, y = [], []
    for i in range(len(a) - w - h + 1):
        X.append(a[i:i + w]); y.append(a[i + w + h - 1])
    return np.array(X), np.array(y)

# Step 6-8: per-machine CHRONOLOGICAL 80/20 split (no shuffling, no leakage)
tr_raw, te_raw = [], []
for s in series:
    k = int(len(s) * 0.8)
    if k <= WINDOW + HORIZON or len(s) - k <= HORIZON:
        continue
    tr_raw.append(s[:k])                                  # first 80% of THIS machine
    te_raw.append(s[max(0, k - WINDOW - HORIZON + 1):])   # tail + context for 1st test window

# Step 9: fit the scaler on TRAINING points ONLY, then transform train & test
scaler = MinMaxScaler().fit(np.concatenate(tr_raw).reshape(-1, 1))
def stack(raws):
    Xs, ys = [], []
    for r in raws:
        rs = scaler.transform(r.reshape(-1, 1)).ravel()
        X, y = make_windows(rs)
        if len(X):
            Xs.append(X); ys.append(y)
    return np.concatenate(Xs)[..., None], np.concatenate(ys)
Xtr, ytr = stack(tr_raw)
Xte, yte = stack(te_raw)
print("windows  train:", Xtr.shape, " test:", Xte.shape)

# Step 10: Model architecture  --  stacked LSTM
model = Sequential([
    Input((WINDOW, 1)),
    LSTM(64, return_sequences=True), Dropout(0.2),
    LSTM(32),                        Dropout(0.2),
    Dense(1),
])

# Step 11: hyperparameters / compilation
model.compile(optimizer=Adam(1e-3), loss=Huber(delta=1.0))
model.summary()

# Step 12: train the model
history = model.fit(
    Xtr, ytr, validation_split=0.1, epochs=30, batch_size=128,
    shuffle=False, verbose=1,
    callbacks=[EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
               ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6)])

# Step 13-14: predictions (inverse-scaled back to [0,1] utilisation)
def inv(a):
    return scaler.inverse_transform(np.asarray(a).reshape(-1, 1)).ravel()
train_pred, test_pred = inv(model.predict(Xtr, verbose=0)), inv(model.predict(Xte, verbose=0))
ytr_i, yte_i = inv(ytr), inv(yte)

# Step 15-17: metrics  (MAE, MAPE, RMSE, R2, sMAPE) for train and test
def smape(y, p):
    return float(np.mean(2*np.abs(p - y) / (np.abs(y) + np.abs(p) + EPS)) * 100)
def scores(y, p):
    return dict(MAE=float(mean_absolute_error(y, p)),
               MAPE=float(mean_absolute_percentage_error(y, p) * 100),
               RMSE=float(np.sqrt(np.mean((y - p) ** 2))),
               R2=float(r2_score(y, p)), sMAPE=smape(y, p))
tr_s, te_s = scores(ytr_i, train_pred), scores(yte_i, test_pred)
print("\nAlibaba 2017  |  LSTM  |  " + UTIL_COL)
print("  TRAIN  MAE {:.4f}  RMSE {:.4f}  R2 {:.4f}  sMAPE {:.1f}".format(
    tr_s["MAE"], tr_s["RMSE"], tr_s["R2"], tr_s["sMAPE"]))
print("  TEST   MAE {:.4f}  RMSE {:.4f}  R2 {:.4f}  sMAPE {:.1f}".format(
    te_s["MAE"], te_s["RMSE"], te_s["R2"], te_s["sMAPE"]))

# Step 18: plot actual vs predicted (first 500 test points)
plt.figure(figsize=(13, 4))
plt.plot(yte_i[:500], label="actual", lw=1)
plt.plot(test_pred[:500], label="predicted", lw=1)
plt.title("Alibaba 2017 - LSTM - " + UTIL_COL + " (test, first 500)")
plt.legend(); plt.tight_layout(); plt.show()

# Step 19: save final results
row = dict(dataset="Alibaba 2017", module="forecasting", target=UTIL_COL, model="LSTM",
           window=WINDOW, horizon=HORIZON, n_train=int(len(Xtr)), n_test=int(len(Xte)),
           train_MAE=tr_s["MAE"], train_MAPE=tr_s["MAPE"], train_RMSE=tr_s["RMSE"],
           train_R2=tr_s["R2"], train_sMAPE=tr_s["sMAPE"],
           test_MAE=te_s["MAE"], test_MAPE=te_s["MAPE"], test_RMSE=te_s["RMSE"],
           test_R2=te_s["R2"], test_sMAPE=te_s["sMAPE"])
os.makedirs(os.path.join(RESULTS_DIR, "sequence"), exist_ok=True)
out_csv = os.path.join(RESULTS_DIR, "sequence", "Alibaba_2017_LSTM_Memory.csv")
pd.DataFrame([row]).to_csv(out_csv, index=False)
print("\nsaved ->", out_csv)